# Historical course notebook
Preserved for provenance. Known split/target issues are documented in ../../docs/LEGACY-REVIEW.md. Outputs are cleared; original scores are not a validated benchmark.


# Bitcoin Analysis

DATA 198 DeCal

## Time Series

In [ ]:
from datetime import datetime
import pandas as pd
import numpy as np
#imports modules for data analysis

In [ ]:
btc_data = pd.read_csv("btc-historical-data.csv") #imported datasets: https://www.kaggle.com/datasets/mczielinski/bitcoin-historical-data 
btc_data["Timestamp"] = pd.to_datetime(btc_data["Timestamp"], unit="s") #converted timestamp to datetime

In [ ]:
btc_data.head()

In [ ]:
#split timestamp into day, month, year, and time-in-date
btc_data["Day"] = btc_data["Timestamp"].dt.day.astype(str) 
btc_data["Month"] = btc_data["Timestamp"].dt.month.astype(str)
btc_data["Year"] = btc_data["Timestamp"].dt.year.astype(str)
btc_data["Time-in-date"] = btc_data["Year"] + "-" + btc_data["Month"] + "-" + btc_data["Day"]
btc_data.head()

In [ ]:
btc_data = btc_data.drop_duplicates(subset=["Time-in-date"]) #drop duplicates
btc_data["close-in-log"] = np.log(btc_data["Close"]) #log transformation of data from exponential to "more stationary"
btc_data.head()

In [ ]:
btc_data.isna().sum() #checks null values in dataset

In [ ]:
btc_data.plot(x="Year", y="close-in-log")
#plots data with Year on x-axis and Close-in-log on y-axis

In [ ]:
btc_data["close-in-log-diff"] = btc_data['close-in-log'].diff()
#log differencing of data
btc_data.plot(x="Year", y="close-in-log-diff")
#plots data with Year on x-axis and close-in-log-diff on y-axis

In [ ]:
#CHECKPOINT 3 HYPOTHESIS TESTING
#note: ours is not a traditional hypothesis testing, but we use an adf test to test for stationarity
#as such, we dont have a specific test-statistic for this, but we can use adf test to get a p-value and we use that to reject or fail to reject the null

#imports modules for time series analysis
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.stattools import adfuller  # The correct function name is 'adfuller', not 'addfuller'

# Before - testing stationarity of the original Close prices
plot_acf(btc_data["Close"].dropna())
plot_pacf(btc_data["Close"].dropna())
#^drops null values
print("Augmented Dickey-Fuller test for Close prices:")
print(adfuller(btc_data["Close"].dropna())[1])
#^tests if data is stationary (not) [null is that the data is not stationary] [alternative is that the data is stationary]
#the p-value is around 0.3 which is >0.05 so the data is not stationary (we fail to reject the null hypothesis)
#thus we have to apply log-differencing

# After - testing stationarity of the log-differenced prices
plot_acf(btc_data["close-in-log-diff"].dropna())
plot_pacf(btc_data["close-in-log-diff"].dropna())
#^hypothesis testing: drops null values
print("\nAugmented Dickey-Fuller test for log-differenced prices:")
print(adfuller(btc_data["close-in-log-diff"].dropna())[1])
#^hypothesis testing: tests if data is stationary (yes) [null is that the data is not stationary] [alternative is that the data is stationary]
#the p-value is a lot smaller than 0.05 so the data is stationary (we successfully reject the null hypothesis)
#this means we can use this data for time series analysis (post log-differencing)


In [ ]:
btc_data = btc_data[btc_data["Year"].astype(float) >= 2017.0]
from statsmodels.tsa.arima.model import ARIMA

testing_data = btc_data[btc_data["Year"].astype(float) >= 2024.0]
training_data = btc_data[btc_data["Year"].astype(float) < 2024.0]

train_data = list(training_data["close-in-log-diff"])
test_data = list(testing_data["close-in-log-diff"])

In [ ]:
test_data

In [ ]:
predictions = []
#I (Alex) wrote this btw, we will reimplement it after spring break with a hopefully simpler method. RMSE of around 2000 is actually quite good though!
diff_list = list(btc_data["Close"])
for i in range(len(test_data)):
    model = ARIMA(train_data, order=(2, 0, 1)) #p,d,q
    model_fit = model.fit()
    print(i)
    output = model_fit.forecast(steps=1)[0]
    output_transformed = np.exp(output + np.log(diff_list[-len(test_data) + i - 1]))
    predictions.append(output_transformed)

    train_data.append(test_data[i]) 


In [ ]:
predictions

In [ ]:
btc_data["close-in-log-diff"]

In [ ]:
from sklearn.metrics import mean_squared_error
np.sqrt(mean_squared_error(btc_data[btc_data["Year"].astype(float) >= 2024.0]["Close"], predictions))


In [ ]:
predictions = []
#I (Alex) wrote this btw, we will reimplement it after spring break with a hopefully simpler method. RMSE of around 2000 is actually quite good though!
diff_list = list(btc_data["Close"])
model2 = ARIMA(train_data, order=(2, 0, 1)) #p,d,q
model2_fit = model2.fit()
for i in range(len(test_data)):
    print(i)
    output = model2_fit.forecast(steps=1)[0]
    output_transformed = np.exp(output + np.log(diff_list[-len(test_data) + i - 1])) #Use predicted value instead of actual value
    predictions.append(output_transformed)
    model2_fit = model2_fit.append([output], refit=False)

In [ ]:
predictions

In [ ]:
np.sqrt(mean_squared_error(btc_data[btc_data["Year"].astype(float) >= 2024.0]["Close"], predictions))

In [ ]:
import pandas as pd

btc_data['predictions'] = [None] * (len(btc_data) - len(test_data)) + predictions
btc_data.tail()

btc_data_2024 = btc_data[btc_data["Year"].astype(float) >= 2024.0]

import matplotlib.pyplot as plt
import matplotlib.dates as mdates

# Format the date string correctly before converting to datetime
def format_date(year, month, day):
    return f"{int(float(year))}-{int(float(month)):02d}-{int(float(day)):02d}"

btc_data['Time-in-date'] = btc_data.apply(lambda x: format_date(x['Year'], x['Month'], x['Day']), axis=1)
btc_data['Time-in-date'] = pd.to_datetime(btc_data['Time-in-date'])

# Plot for all data
plt.figure(figsize=(12,8))
btc_data.set_index('Time-in-date', inplace=True)
btc_data[['Close','predictions']].plot(alpha=0.6)
plt.gca().xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d'))
plt.gca().xaxis.set_major_locator(mdates.MonthLocator(interval=6))  # Show ticks every 6 months
plt.gcf().autofmt_xdate()  # Rotate and align the tick labels
plt.title('Bitcoin Price Prediction vs Actual (All Data)')
plt.xlabel('Date')
plt.ylabel('Price (USD)')
plt.legend(['Actual Price', 'Predicted Price'])
plt.grid(True)
plt.show()

# Plot for 2024 data
plt.figure(figsize=(12,8))
# Format dates for 2024 data
btc_data_2024['Time-in-date'] = btc_data_2024.apply(lambda x: format_date(x['Year'], x['Month'], x['Day']), axis=1)
btc_data_2024['Time-in-date'] = pd.to_datetime(btc_data_2024['Time-in-date'])
btc_data_2024.set_index('Time-in-date', inplace=True)
btc_data_2024[['Close','predictions']].plot(alpha=0.6)
plt.gca().xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d'))
plt.gca().xaxis.set_major_locator(mdates.MonthLocator(interval=1))  # Show ticks for each month
plt.gcf().autofmt_xdate()  # Rotate and align the tick labels
plt.title('Bitcoin Price Prediction vs Actual (2024)')
plt.xlabel('Date')
plt.ylabel('Price (USD)')
plt.legend(['Actual Price', 'Predicted Price'])
plt.grid(True)
plt.show()

In [ ]:
btc_data.tail()

In [ ]:
predictions = []
#I (Alex) wrote this btw, we will reimplement it after spring break with a hopefully simpler method. RMSE of around 2000 is actually quite good though!
diff_list = list(btc_data["Close"])
model2 = ARIMA(train_data, order=(2, 0, 1)) #p,d,q
model2_fit = model2.fit()
for i in range(len(test_data)):
    print(i)
    output = model2_fit.forecast(steps=1)[0]
    output_transformed = np.exp(output + np.log(diff_list[-len(test_data) + i - 1]))
    predictions.append(output_transformed)
    model2_fit = model2_fit.append([test_data[i]], refit=False)

In [ ]:
btc_data

In [ ]:
# // Three day forecast - credit to the Alex Zhai
# // Source: https://colab.research.google.com/drive/1JIF-kEOB0QYiC85SU2Rx9izcSOCJfvTj?usp=sharing 

diff2 = btc_data[["close-in-log"]].diff(periods=2).dropna()
diff3 = btc_data[["close-in-log"]].diff(periods=3).dropna()

btc_data["Log and Diff 2"] = diff2
btc_data["Log and Diff 3"] = diff3

In [ ]:
btc_data.head()

In [ ]:
train_amount = int(len(btc_data)*0.8)
test_amount = len(btc_data) - train_amount

train1 = btc_data.iloc[0: train_amount]["close-in-log-diff"] #.iloc, the rows and then the columns
test1 = btc_data.iloc[train_amount:train_amount + test_amount]["close-in-log-diff"]

train2 = btc_data.iloc[0: train_amount]["Log and Diff 2"].dropna() #.iloc, the rows and then the columns
test2 = btc_data.iloc[train_amount:train_amount + test_amount]["Log and Diff 2"]

train3 = btc_data.iloc[0: train_amount]["Log and Diff 3"].dropna() #.iloc, the rows and then the columns
test3 = btc_data.iloc[train_amount:train_amount + test_amount]["Log and Diff 3"]

num_obs = len(test3)
len(test3)

model1 = ARIMA(train1, order=(2,0,1)).fit()
model2 = ARIMA(train2, order=(2, 0, 1)).fit()
model3 = ARIMA(train3, order = (2,0,1)).fit()

diff_list = list(btc_data["High"])

predictions2 = []

for i in range(num_obs):

    if i % 3 == 0:  # Model 1 handles t+1, t+4, t+7, ...
        forecast1 = model1.forecast(steps=1).iloc[0]
        prev1 = np.log(diff_list[-len(test1) + i - 1]) #Reversing the differencing
        reversed_pred1 = np.exp(forecast1 + prev1)
        predictions2.append(reversed_pred1)
        model1 = model1.append([list(test1)[i]], refit=False)

    elif i % 3 == 1:  # Model 2 handles t+2, t+5, ...
        forecast2 = model2.forecast(steps=1).iloc[0]
        prev2 = np.log(diff_list[-len(test2) + i - 2])
        reversed_pred2 = np.exp(forecast2 + prev2)
        predictions2.append(reversed_pred2)
        model2 = model2.append([list(test2)[i]], refit=False)

    else:  # i % 3 == 2 → Model 3 handles t+3, t+6, ...
        forecast3 = model3.forecast(steps=1).iloc[0]
        prev3 = np.log(diff_list[-len(test3) + i - 3])
        reversed_pred3 = np.exp(forecast3 + prev3)
        predictions2.append(reversed_pred3)
        model3 = model3.append([list(test3)[i]], refit=False)

    print(i)


predictions2

from sklearn.metrics import mean_squared_error

btc_data["3 Day Predictions"] = [None] * (len(train1)) + predictions2

np.sqrt(mean_squared_error(predictions2, btc_data.iloc[train_amount:train_amount + test_amount]["High"]))

btc_data[["High", "3 Day Predictions"]].plot()

In [ ]:

np.sqrt(mean_squared_error(predictions2, btc_data.iloc[train_amount:train_amount + test_amount]["High"]))

In [ ]:
from sklearn.metrics import mean_squared_error

btc_data["3 Day Predictions"] = [None] * (len(train1)) + predictions2

np.sqrt(mean_squared_error(predictions2, btc_data.iloc[train_amount:train_amount + test_amount]["High"]))

btc_data[["High", "3 Day Predictions"]].plot()
plt.ylabel("Price")
plt.xlabel("Time")
plt.title("Bitcoin Price 3 Day Prediction")

In [ ]:
btc_data.head()

In [ ]:
btc_data[btc_data["Timestamp"].dt.year >= 2024][["High", "3 Day Predictions"]].plot()

In [ ]:
np.sqrt(mean_squared_error(predictions2, btc_data.iloc[train_amount:train_amount + test_amount]["High"]))